# Results 4 — Variant-level pleiotropy

Cluster counts, the negative-binomial model of the variant pleiotropy score (vPS), the
directionality analysis, and the data behind Figure 3.

| file | panel |
| --- | --- |
| `plot_a.csv` | observed and predicted vPS per MAF bin (Figure 3a) |
| `plot_b.csv` | univariate and joint model coefficients (Figure 3b) |
| `figure_3_apoe.csv` | every disease association of the two APOE variants (Figure 3c) |

Predicted power assumes the variant acts on most traits at an effect size an order of
magnitude below its largest observed effect: the non-centrality parameter is
`maxAbsBeta^2 * maxEffectiveSampleSize * maxVarG / 11`, and power is the survival function of
a non-central chi-square at the genome-wide threshold.

In [1]:
import collections

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import ncx2, spearmanr

from manuscript_methods import clusters, paper

numbers = {}

COVARIATES = [
    "maxAbsBetaNormalised",
    "maxMAFNormalised",
    "maxEffectiveSampleSizeNormalised",
    "gerpNormalisedNormalised",
    "vepBinaryNormalised",
    "predictedPowerNormalised",
]
LABELS = {
    "maxAbsBetaNormalised": "Absolute beta",
    "maxMAFNormalised": "MAF",
    "maxEffectiveSampleSizeNormalised": "Sample size",
    "gerpNormalisedNormalised": "GERP",
    "vepBinaryNormalised": "PAV",
    "predictedPowerNormalised": "Predicted power",
}
MAF_BINS = [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
MAF_BIN_LABELS = ["0-0.01", "0.01-0.05", "0.05-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5"]
GENOME_WIDE_CHI2 = 32.84125  # chi-square value corresponding to p = 5e-8 with 1 degree of freedom

## Clusters

In [2]:
cluster_table = pd.read_parquet(paper.derived("variant_clusters"))

numbers["R4.01"] = len(cluster_table)
numbers["R4.02"] = int((cluster_table["uniqueLeadVariants"] > 1).sum())
numbers["R4.03"] = int((cluster_table["uniqueDiseases"] > 1).sum())
numbers["R4.04"] = int(cluster_table["uniqueDiseases"].max())
numbers["R4.05"] = round(float(cluster_table["uniqueDiseases"].mean()), 2)
numbers["R4.06"] = int((cluster_table["uniqueTherapeuticAreas"] > 1).sum())
numbers["R4.07"] = int(cluster_table["uniqueTherapeuticAreas"].max())
numbers["R4.08"] = round(float(cluster_table["uniqueTherapeuticAreas"].mean()), 2)
rho, pvalue = spearmanr(cluster_table["uniqueDiseases"], cluster_table["uniqueTherapeuticAreas"])
numbers["R4.09"] = round(float(rho), 2)
print({k: numbers[k] for k in sorted(numbers)}, "| Spearman P:", pvalue)

{'R4.01': 20041, 'R4.02': 5595, 'R4.03': 6617, 'R4.04': 120, 'R4.05': 2.14, 'R4.06': 4766, 'R4.07': 19, 'R4.08': 1.45, 'R4.09': 0.84} | Spearman P: 0.0


In [3]:
# The cluster partition and the pleiotropy score itself must not move under the redefinition:
# these are the values the unchanged pipeline produced, asserted rather than merely printed.
CONTROL = {
    "R4.01": 20041,
    "R4.02": 5595,
    "R4.03": 6617,
    "R4.04": 120,
    "R4.05": 2.14,
    "R4.06": 4766,
    "R4.07": 19,
    "R4.08": 1.45,
    "R4.09": 0.84,
}
for key, expected in CONTROL.items():
    assert numbers[key] == expected, (key, numbers[key], expected)
print("cluster and vPS statistics unchanged:", len(CONTROL), "values")

cluster and vPS statistics unchanged: 9 values


## Cluster-level covariates

Each cluster is represented by the lead variant of its **seed** credible set: the member with the
smallest association P value, ties broken by the higher lead-variant PIP.
`clusters.load_credible_sets` sorts on exactly that and `clusters.cluster` seeds each component
from the first member it reaches in that order, so the seed is the most significant credible set of
its cluster.

This replaces the previous rule, which picked the lead variant associated with the most diseases in
the cluster (ties broken by variant id). That rule chose the covariate carrier by an outcome
correlated with vPS itself; the seed rule chooses it by association strength, independently of how
many diseases the cluster spans. `mostDiseasesVariantId` is kept beside it so the two can be
compared, and the models below are fitted on the seed representative.

In [4]:
credible_sets = clusters.load_credible_sets()
edges = clusters.load_edges(set(credible_sets["studyLocusId"]))
components = clusters.cluster(list(zip(credible_sets["studyLocusId"], credible_sets["variantId"])), edges)

locus_variant = dict(zip(credible_sets["studyLocusId"], credible_sets["variantId"]))
locus_traits = dict(zip(credible_sets["studyLocusId"], credible_sets["diseaseIds"]))

# `uniqueDiseases` and `betaSignConcordance` are the published columns, kept so the earlier lineage
# stays reproducible; `leadVPS` / `leadDirectionalConcordance` are the first redefinition and
# `signedLeadVPS` / `signedLeadDirectionalConcordance` the amended, sign-gated pair. Only the model
# covariates above are used below; the score columns are carried for Supplementary Figures SR 2 and
# SR 3, which draw one point per cluster at its representative lead variant.
features = pd.read_parquet(paper.derived("variant_features"))[
    [
        "variantId",
        "maxAbsBeta",
        "maxMAF",
        "maxEffectiveSampleSize",
        "maxVarG",
        "gerpNormalised",
        "vepScore",
        "uniqueDiseases",
        "betaSignConcordance",
        "leadVPS",
        "leadVPSDefined",
        "leadDirectionalConcordance",
        "leadConcordanceDefined",
        "signedLeadVPS",
        "signedLeadVPSDefined",
        "signedLeadDirectionalConcordance",
    ]
].drop_duplicates("variantId")

rows = []
for seed, members in components:
    per_variant = collections.defaultdict(set)
    for locus_id in members:
        per_variant.setdefault(locus_variant[locus_id], set())
        traits = locus_traits.get(locus_id)
        if traits is not None:
            per_variant[locus_variant[locus_id]].update(traits)
    all_traits = set().union(*per_variant.values()) if per_variant else set()
    rows.append(
        {
            "clusterSize": len(members),
            "vPS": len(all_traits),
            # the seed is the most significant credible set of the cluster
            "clusterVariantId": locus_variant[seed],
            # the previous rule, carried for comparison only
            "mostDiseasesVariantId": sorted(per_variant.items(), key=lambda kv: (-len(kv[1]), kv[0]))[0][0],
        }
    )

frame = pd.DataFrame(rows).merge(features, left_on="clusterVariantId", right_on="variantId", how="inner")
frame.to_parquet(paper.derived("cluster_covariates"), index=False)
print("clusters joined to variant features:", len(frame), "of", len(components))

# vPS is computed from the cluster's whole disease set, so it cannot depend on the representative.
control = pd.read_parquet(paper.derived("variant_clusters"))
assert len(frame) == len(control) == numbers["R4.01"]
assert frame["vPS"].tolist() == control["uniqueDiseases"].tolist()
assert frame["clusterSize"].tolist() == control["clusterSize"].tolist()
assert frame["clusterVariantId"].tolist() == control["leadVariantId"].tolist()
print("vPS, cluster sizes and the seed lead variant agree with `variant_clusters`")

changed = int((frame["clusterVariantId"] != frame["mostDiseasesVariantId"]).sum())
multi = int((control["uniqueLeadVariants"] > 1).sum())
print(
    f"representative differs from the most-diseases variant in {changed:,} of {len(frame):,} clusters "
    f"({100 * changed / len(frame):.1f}%); {changed:,} of the {multi:,} clusters that have more than "
    f"one lead variant ({100 * changed / multi:.1f}%)"
)

clusters joined to variant features: 20041 of 20041
vPS, cluster sizes and the seed lead variant agree with `variant_clusters`
representative differs from the most-diseases variant in 3,292 of 20,041 clusters (16.4%); 3,292 of the 5,595 clusters that have more than one lead variant (58.8%)


In [5]:
frame["ncp"] = (frame["maxAbsBeta"] ** 2 * frame["maxEffectiveSampleSize"] * frame["maxVarG"]) / 11
frame["predictedPower"] = ncx2.sf(x=GENOME_WIDE_CHI2, df=1, nc=frame["ncp"])
frame["gerpNormalised"] = frame["gerpNormalised"].fillna(frame["gerpNormalised"].mean())
frame["vepBinary"] = (frame["vepScore"] >= 0.66).astype(int)

# Every covariate is min-max scaled so the coefficients are comparable in the forest plot.
for column in ["maxAbsBeta", "maxMAF", "gerpNormalised", "vepBinary", "maxEffectiveSampleSize", "predictedPower"]:
    span = frame[column].max() - frame[column].min()
    frame[f"{column}Normalised"] = 0.0 if span == 0 else (frame[column] - frame[column].min()) / span

## Figure 3b — negative binomial models

In [6]:
def fit(covariates):
    """Negative binomial fit of vPS on the given covariates."""
    x = sm.add_constant(frame[covariates].copy())
    model = sm.NegativeBinomial(frame["vPS"], x).fit(disp=False, maxiter=1000)
    return model, x


records = []
for covariate in COVARIATES:
    model, _ = fit([covariate])
    ci = model.conf_int()
    records.append(
        {
            "covariate": covariate,
            "model_type": "Univariate",
            "coefficient": model.params[covariate],
            "std_error": model.bse[covariate],
            "p_value": model.pvalues[covariate],
            "ci_lower": ci.loc[covariate, 0],
            "ci_upper": ci.loc[covariate, 1],
        }
    )

joint, x_joint = fit(COVARIATES)
ci = joint.conf_int()
for covariate in COVARIATES:
    records.append(
        {
            "covariate": covariate,
            "model_type": "Multi",
            "coefficient": joint.params[covariate],
            "std_error": joint.bse[covariate],
            "p_value": joint.pvalues[covariate],
            "ci_lower": ci.loc[covariate, 0],
            "ci_upper": ci.loc[covariate, 1],
        }
    )

coefficients = pd.DataFrame(records)
coefficients["covariate_label"] = coefficients["covariate"].map(LABELS)
coefficients["y_numerical"] = coefficients["covariate"].map({c: i for i, c in enumerate(COVARIATES)})
coefficients["y_plot"] = coefficients["y_numerical"] + np.where(coefficients["model_type"] == "Univariate", -0.1, 0.1)
coefficients = coefficients[
    [
        "covariate",
        "covariate_label",
        "model_type",
        "coefficient",
        "std_error",
        "p_value",
        "ci_lower",
        "ci_upper",
        "y_numerical",
        "y_plot",
    ]
]
coefficients.to_csv(paper.derived("plot_b.csv"), index=False)

# The sign gate touches lead_vPS only. The representative is chosen by smallest association P value
# and the model outcome is the cluster's vPS over every disease term, so neither the design matrix
# nor the outcome can move; asserted rather than assumed, so Figure 3 does not have to be rebuilt.
FIGURE_3B = {
    ("maxAbsBetaNormalised", "Univariate"): 1.8197915326,
    ("maxMAFNormalised", "Univariate"): 0.3752694380,
    ("maxEffectiveSampleSizeNormalised", "Univariate"): 0.7762604161,
    ("gerpNormalisedNormalised", "Univariate"): 0.2256798221,
    ("vepBinaryNormalised", "Univariate"): 0.6007236108,
    ("predictedPowerNormalised", "Univariate"): 1.4423452543,
    ("maxAbsBetaNormalised", "Multi"): 0.3822882061,
    ("maxMAFNormalised", "Multi"): 0.3088470437,
    ("maxEffectiveSampleSizeNormalised", "Multi"): 0.3064897302,
    ("gerpNormalisedNormalised", "Multi"): 0.0284565663,
    ("vepBinaryNormalised", "Multi"): 0.2139844400,
    ("predictedPowerNormalised", "Multi"): 1.3639763290,
}
for (covariate, model_type), expected in FIGURE_3B.items():
    got = float(
        coefficients.loc[
            (coefficients["covariate"] == covariate) & (coefficients["model_type"] == model_type), "coefficient"
        ].iloc[0]
    )
    assert abs(got - expected) < 1e-6, (covariate, model_type, got, expected)
print("all twelve Figure 3b coefficients unchanged")
coefficients.round(4)

all twelve Figure 3b coefficients unchanged


,covariate,covariate_label,model_type,coefficient,std_error,p_value,ci_lower,ci_upper,y_numerical,y_plot
0,maxAbsBetaNormalised,Absolute beta,Univariate,1.8198,0.0752,0.0000,1.6725,1.9671,0,-0.1
1,maxMAFNormalised,MAF,Univariate,0.3753,0.0231,0.0000,0.3301,0.4205,1,0.9
2,maxEffectiveSampleSizeNormalised,Sample size,Univariate,0.7763,0.0370,0.0000,0.7037,0.8488,2,1.9
3,gerpNormalisedNormalised,GERP,Univariate,0.2257,0.0280,0.0000,0.1708,0.2806,3,2.9
4,vepBinaryNormalised,PAV,Univariate,0.6007,0.0345,0.0000,0.5331,0.6683,4,3.9
5,predictedPowerNormalised,Predicted power,Univariate,1.4423,0.0152,0.0000,1.4125,1.4722,5,4.9
6,maxAbsBetaNormalised,Absolute beta,Multi,0.3823,0.0757,0.0000,0.2340,0.5306,0,0.1
7,maxMAFNormalised,MAF,Multi,0.3088,0.0228,0.0000,0.2643,0.3534,1,1.1
8,maxEffectiveSampleSizeNormalised,Sample size,Multi,0.3065,0.0340,0.0000,0.2399,0.3731,2,2.1
9,gerpNormalisedNormalised,GERP,Multi,0.0285,0.0269,0.2895,-0.0242,0.0811,3,3.1


## Variance explained

In [7]:
without_power = [c for c in COVARIATES if c != "predictedPowerNormalised"]
joint_no_power, x_no_power = fit(without_power)
power_only, x_power = fit(["predictedPowerNormalised"])
sample_size_only, x_sample = fit(["maxEffectiveSampleSizeNormalised"])


def r2(model, x):
    """Squared Pearson correlation between observed and predicted vPS."""
    return float(np.corrcoef(frame["vPS"], model.predict(x))[0, 1] ** 2)


numbers["R4.10"] = 100 * r2(power_only, x_power)
numbers["R4.11"] = 100 * r2(joint, x_joint)
numbers["R4.12"] = 100 * r2(joint_no_power, x_no_power)
numbers["R4.13"] = 100 * r2(sample_size_only, x_sample)
print({k: numbers[k] for k in ["R4.10", "R4.11", "R4.12", "R4.13"]})

# Same argument as the Figure 3b coefficients: the gate cannot reach the model.
VARIANCE_EXPLAINED = {
    "R4.10": 15.073413831782467,
    "R4.11": 16.946450149990614,
    "R4.12": 3.826953106634868,
    "R4.13": 0.5182255329135724,
}
for key, expected in VARIANCE_EXPLAINED.items():
    assert abs(numbers[key] - expected) < 1e-9, (key, numbers[key], expected)
print("R4.10-R4.13 unchanged")

{'R4.10': 15.073413831782467, 'R4.11': 16.946450149990614, 'R4.12': 3.826953106634868, 'R4.13': 0.5182255329135724}
R4.10-R4.13 unchanged


## Figure 3a — observed and predicted vPS per MAF bin

In [8]:
binned = frame.copy()
binned["predicted_traits_full_model"] = joint.predict(x_joint)
binned["predicted_traits_no_power"] = joint_no_power.predict(x_no_power)
binned["maxMAF_bin"] = pd.cut(binned["maxMAF"], bins=MAF_BINS, labels=MAF_BIN_LABELS, right=False)

bins = (
    binned.groupby("maxMAF_bin", observed=False)
    .agg(
        observed_mean=("vPS", "mean"),
        observed_sem=("vPS", "sem"),
        predicted_full_mean=("predicted_traits_full_model", "mean"),
        predicted_full_sem=("predicted_traits_full_model", "sem"),
        predicted_no_power_mean=("predicted_traits_no_power", "mean"),
        predicted_no_power_sem=("predicted_traits_no_power", "sem"),
    )
    .reset_index()
)
bins["maxMAF_bin"] = bins["maxMAF_bin"].astype(str)
bins.to_csv(paper.derived("plot_a.csv"), index=False)
bins.round(4)

,maxMAF_bin,observed_mean,observed_sem,predicted_full_mean,predicted_full_sem,predicted_no_power_mean,predicted_no_power_sem
0,0-0.01,1.9294,0.1328,2.2990,0.0755,3.1397,0.1101
1,0.01-0.05,1.7555,0.0602,1.8516,0.0273,1.9734,0.0263
2,0.05-0.1,1.8956,0.0714,1.8690,0.0305,1.7886,0.0188
3,0.1-0.2,2.0263,0.0644,1.9208,0.0212,1.8078,0.0097
4,0.2-0.3,2.1457,0.0542,2.0897,0.0225,2.0227,0.0101
5,0.3-0.4,2.2628,0.0556,2.2657,0.0239,2.2826,0.0106
6,0.4-0.5,2.4863,0.0816,2.4877,0.0258,2.5727,0.0106


## Directionality

Over the cluster representatives, since lead_vPS is reported on them.

A credible set contributes to `signedLeadVPS` only if the variant is its lead variant, its study
maps to exactly one disease term, **and** `rescaledStatistics.directionOfEffect` is non-null. That
last gate is the amendment: the column is null exactly where only an absolute effect size could be
obtained, which carries no directional information, so such a disease must not be counted. See
`01-data-preparation/07_variant_features` for the size of the gate.

Because every counted disease is signed by construction, concordance is computable wherever
`signedLeadVPS` is at least 1 and equals 1 where it is 1. **There are therefore only two groups**:
defined, and excluded because nothing contributes at all. The previous three-group split is
recomputed beside it, as is the published `betaSignConcordance` reading, so all three lineages stay
visible.

In [9]:
variant_features = pd.read_parquet(paper.derived("variant_features"))
representatives = variant_features[variant_features["variantId"].isin(set(frame["clusterVariantId"]))]
print("cluster representatives:", len(representatives))

# Excluded outright: no signed contributing credible set, so no lead_vPS. Never counted as 0.
numbers["R4.14x"] = int((~representatives["signedLeadVPSDefined"]).sum())

pleiotropic = representatives[representatives["signedLeadVPSDefined"] & (representatives["signedLeadVPS"] > 1)]
concordant = pleiotropic[pleiotropic["signedLeadDirectionalConcordance"] == 1.0]
discordant = pleiotropic[pleiotropic["signedLeadDirectionalConcordance"] < 1.0]

# No third group can survive the gate: every counted disease is signed, so the concordance of a
# variant with signedLeadVPS >= 1 is always computable.
assert representatives.loc[representatives["signedLeadVPSDefined"], "signedLeadDirectionalConcordance"].notna().all()
assert len(concordant) + len(discordant) == len(pleiotropic)

numbers["R4.14"] = len(pleiotropic)
numbers["R4.15"] = len(concordant)
numbers["R4.16"] = len(discordant)

highly_pleiotropic = pleiotropic[pleiotropic["signedLeadVPS"] >= 10]
low_agreement = highly_pleiotropic[highly_pleiotropic["signedLeadDirectionalConcordance"] <= 0.8]
numbers["R4.17"] = len(highly_pleiotropic)
numbers["R4.18"] = len(low_agreement)
numbers["R4.19"] = len(
    {gene for genes in low_agreement["prioritisedGenes"] for gene in (genes if genes is not None else [])}
)

print({k: numbers[k] for k in ["R4.14", "R4.15", "R4.16", "R4.14x", "R4.17", "R4.18", "R4.19"]})
print("fully concordant share: %.1f%%" % (100 * len(concordant) / len(pleiotropic)))


def lineage(vps, concordance, label):
    """The same block under one of the earlier definitions, for comparison."""
    universe = representatives[representatives[vps].notna() & (representatives[vps] > 1)]
    defined = universe[universe[concordance].notna()]
    high = defined[defined[vps] >= 10]
    low = high[high[concordance] <= 0.8]
    return {
        "definition": label,
        "excluded (no lead_vPS)": int(representatives[vps].isna().sum()),
        "pleiotropic": len(universe),
        "concordance 1": int((defined[concordance] == 1.0).sum()),
        "concordance < 1": int((defined[concordance] < 1.0).sum()),
        "concordance undefined": len(universe) - len(defined),
        ">= 10 diseases": len(high),
        "<= 0.8": len(low),
        "genes": len({g for gs in low["prioritisedGenes"] for g in (gs if gs is not None else [])}),
    }


print()
print(
    pd.DataFrame(
        [
            lineage("signedLeadVPS", "signedLeadDirectionalConcordance", "amended, sign-gated"),
            lineage("leadVPS", "leadDirectionalConcordance", "ungated leadVPS"),
            lineage("uniqueDiseases", "betaSignConcordance", "published"),
        ]
    )
    .set_index("definition")
    .T.to_string()
)

cluster representatives: 20041
{'R4.14': 2166, 'R4.15': 1844, 'R4.16': 322, 'R4.14x': 3019, 'R4.17': 67, 'R4.18': 18, 'R4.19': 21}
fully concordant share: 85.1%

definition              amended, sign-gated  ungated leadVPS  published
excluded (no lead_vPS)                 3019             1250          0
pleiotropic                            2166             2341       3983
concordance 1                          1844             1774       3002
concordance < 1                         322              307        455
concordance undefined                     0              260        526
>= 10 diseases                           67               68         91
<= 0.8                                   18               20         23
genes                                    21               23         25


In [10]:
# Supplementary Table 2: the lead variants with a high lead_vPS and low directional agreement.
# The universe is the cluster representatives, the same one the sentence above is computed on;
# `06-supplementary-tables/01` builds the published sheet over every lead variant instead.
st2 = low_agreement.sort_values(["signedLeadVPS", "variantId"], ascending=[False, True])[
    [
        "variantId",
        "prioritisedGenes",
        "signedLeadVPS",
        "signedLeadUniqueTherapeuticAreas",
        "signedLeadDirectionalConcordance",
        "leadVPS",
        "leadDirectionalConcordance",
        "uniqueDiseases",
        "uniqueTherapeuticAreas",
        "betaSignConcordance",
        "maxMAF",
    ]
]
st2.to_csv(paper.derived("st2_discordant_variants.csv"), index=False)
print("rows:", len(st2))
st2.head(10)

rows: 18


,variantId,prioritisedGenes,signedLeadVPS,signedLeadUniqueTherapeuticAreas,signedLeadDirectionalConcordance,leadVPS,leadDirectionalConcordance,uniqueDiseases,uniqueTherapeuticAreas,betaSignConcordance,maxMAF
22854,2_27508073_T_C,[ENSG00000084734],32.0,13.0,0.656250,33.0,0.677419,34,13,0.627660,0.471640
14429,7_5397122_C_T,[ENSG00000182095],26.0,8.0,0.769231,26.0,0.769231,43,9,0.811321,0.042946
22467,22_28725099_A_G,[ENSG00000183765],23.0,4.0,0.739130,23.0,0.739130,32,4,0.694444,0.025757
13951,6_12903725_A_G,[ENSG00000112137],20.0,3.0,0.600000,20.0,0.600000,20,3,0.685393,0.443330
35647,12_4275678_T_G,[ENSG00000118971],17.0,11.0,0.647059,17.0,0.625000,20,12,0.775000,0.027176
754,12_57133500_T_C,[ENSG00000123384],16.0,4.0,0.625000,16.0,0.625000,16,4,0.756757,0.401119
36698,19_19268740_C_T,"[ENSG00000213996, ENSG00000129933]",16.0,6.0,0.750000,16.0,0.750000,16,6,0.729730,0.074398
29200,6_90267049_G_A,[ENSG00000112182],15.0,4.0,0.533333,16.0,0.533333,18,4,0.600000,0.165131
6122,14_94371805_G_T,"[ENSG00000197249, ENSG00000258597]",14.0,9.0,0.714286,14.0,0.714286,17,9,0.769231,0.020748
30575,12_111269073_C_T,"[ENSG00000111249, ENSG00000111252]",13.0,7.0,0.615385,13.0,0.615385,13,7,0.714286,0.301200


## The two APOE variants of Figure 3c

In [11]:
APOE_VARIANTS = ["19_44908684_T_C", "19_44908822_C_T"]

apoe = variant_features[variant_features["variantId"].isin(APOE_VARIANTS)].set_index("variantId")
lead = apoe.loc["19_44908684_T_C"]
numbers["R4.20"] = int(lead["signedLeadVPS"])
numbers["R4.21"] = round(float(lead["signedLeadDirectionalConcordance"]), 2)
numbers["R4.22"] = int(lead["signedLeadUniqueTherapeuticAreas"])
print({k: numbers[k] for k in ["R4.20", "R4.21", "R4.22"]})
print(
    pd.DataFrame(
        [
            {
                "definition": "amended, sign-gated",
                "lead_vPS": int(lead["signedLeadVPS"]),
                "concordance": round(float(lead["signedLeadDirectionalConcordance"]), 4),
                "therapeutic areas": int(lead["signedLeadUniqueTherapeuticAreas"]),
                "up": int(lead["signedLeadUpDiseases"]),
                "down": int(lead["signedLeadDownDiseases"]),
            },
            {
                "definition": "ungated leadVPS",
                "lead_vPS": int(lead["leadVPS"]),
                "concordance": round(float(lead["leadDirectionalConcordance"]), 4),
                "therapeutic areas": int(lead["leadUniqueTherapeuticAreas"]),
                "up": int(lead["leadUpDiseases"]),
                "down": int(lead["leadDownDiseases"]),
            },
            {
                "definition": "published",
                "lead_vPS": int(lead["uniqueDiseases"]),
                "concordance": round(float(lead["betaSignConcordance"]), 4),
                "therapeutic areas": int(lead["uniqueTherapeuticAreas"]),
                "up": None,
                "down": None,
            },
        ]
    )
    .set_index("definition")
    .to_string()
)

# Still the most pleiotropic lead variant under the amendment?
ranked = variant_features.loc[variant_features["signedLeadVPS"].notna(), ["variantId", "signedLeadVPS"]]
ranked = ranked.sort_values(["signedLeadVPS", "variantId"], ascending=[False, True]).head(5)
print("\ntop five lead variants by signedLeadVPS:")
print(ranked.to_string(index=False))

{'R4.20': 71, 'R4.21': 0.56, 'R4.22': 15}
                     lead_vPS  concordance  therapeutic areas    up  down
definition                                                               
amended, sign-gated        71       0.5634                 15  40.0  31.0
ungated leadVPS            71       0.5714                 15  40.0  30.0
published                  85       0.6595                 15   NaN   NaN

top five lead variants by signedLeadVPS:
       variantId  signedLeadVPS
 19_44908684_T_C           71.0
10_112998590_C_T           47.0
 1_113834946_A_G           41.0
 4_102267552_C_T           38.0
 6_160589086_A_G           38.0


In [12]:
import pyarrow.compute as pc
import pyarrow.dataset as ds

names = clusters.disease_names()
areas = clusters.therapeutic_area_lookup()

associations = (
    ds.dataset(paper.derived("qualifying_credible_sets"), format="parquet")
    .to_table(
        columns={
            "variantId": ds.field("variantId"),
            "studyId": ds.field("studyId"),
            "diseaseIds": ds.field("diseaseIds"),
            "originalBeta": ds.field("originalBeta"),
            "estimatedBeta": pc.struct_field(ds.field("rescaledStatistics"), "minorAlleleEstimatedBeta"),
            "absEstimatedBeta": pc.struct_field(ds.field("rescaledStatistics"), "absEstimatedBeta"),
            "directionOfEffect": pc.struct_field(ds.field("rescaledStatistics"), "directionOfEffect"),
            "pValueMantissa": pc.struct_field(ds.field("variantStatistics"), "pValueMantissa"),
            "pValueExponent": pc.struct_field(ds.field("variantStatistics"), "pValueExponent"),
        },
        filter=pc.field("variantId").isin(APOE_VARIANTS),
    )
    .to_pandas()
)
associations = associations[associations["originalBeta"].notna()].copy()

# The rescaled magnitude carrying the harmonised sign: the same quantity as
# `minorAlleleEstimatedBeta` without the minor-allele flip. `contributing` marks the credible sets
# lead_vPS and the concordance are computed on.
associations["harmonisedEstimatedBeta"] = associations["directionOfEffect"] * associations["absEstimatedBeta"]
associations["contributing"] = associations["diseaseIds"].map(lambda ids: ids is not None and len(ids) == 1)
# The amended contributing set adds the sign gate. Panel c is still drawn from `contributing`, so
# the committed figure stands; the difference is reported rather than acted on.
associations["signedContributing"] = associations["contributing"] & associations["directionOfEffect"].notna()
associations["diseaseNames"] = associations["diseaseIds"].map(
    lambda ids: [names.get(d) for d in (ids if ids is not None else [])]
)
associations["mappedTherapeuticAreas"] = associations["diseaseIds"].map(
    lambda ids: sorted({areas.get(d, "other") for d in (ids if ids is not None else [])})
)
associations["therapeuticAreaNames"] = associations["mappedTherapeuticAreas"].map(
    lambda tas: [paper.THERAPEUTIC_AREAS.get(t, "other") for t in tas]
)
associations["neg_log10_p"] = -(np.log10(associations["pValueMantissa"]) + associations["pValueExponent"])

columns = [
    "studyId",
    "diseaseIds",
    "diseaseNames",
    "mappedTherapeuticAreas",
    "therapeuticAreaNames",
    "estimatedBeta",
    "harmonisedEstimatedBeta",
    "contributing",
    "pValueMantissa",
    "pValueExponent",
    "neg_log10_p",
]
outputs = ["variant_pleiotropy_data_exploded.csv", "variant_pleiotropy_data_exploded_2.csv"]
for variant, name in zip(APOE_VARIANTS, outputs):
    subset = associations[associations["variantId"] == variant].explode("therapeuticAreaNames")
    subset[columns].to_csv(paper.derived(name), index=False)
    print(
        name,
        subset.shape,
        "| contributing rows:",
        int(subset["contributing"].sum()),
        "| signed contributing rows:",
        int(subset["signedContributing"].sum()),
    )

# Does the nine-negative / six-positive therapeutic-area split of the APOE sentence survive the
# gate? One row per contributing disease, at its most significant association, majority direction
# per therapeutic area.
signed = associations[(associations["variantId"] == "19_44908684_T_C") & associations["signedContributing"]].copy()
signed["diseaseId"] = signed["diseaseIds"].map(lambda ids: ids[0])
signed = signed.sort_values(["pValueExponent", "pValueMantissa"]).drop_duplicates("diseaseId")
# The legacy hierarchy column, because that is what the variant-level area count uses.
legacy_areas = clusters.therapeutic_area_lookup("primaryTherapeuticAreaLegacy")
signed["therapeuticArea"] = signed["diseaseId"].map(lambda d: legacy_areas.get(d, "other"))
by_area = signed.groupby("therapeuticArea")["directionOfEffect"].agg(
    diseases="size", up=lambda s: int((s > 0).sum()), down=lambda s: int((s < 0).sum())
)
by_area["majority"] = np.where(
    by_area["up"] > by_area["down"], "positive", np.where(by_area["down"] > by_area["up"], "negative", "tied")
)
by_area.index = [paper.THERAPEUTIC_AREAS.get(a, a) for a in by_area.index]
print()
print(by_area.to_string())
print("therapeutic-area majority split:", by_area["majority"].value_counts().to_dict())

variant_pleiotropy_data_exploded.csv (193, 16) | contributing rows: 170 | signed contributing rows: 170
variant_pleiotropy_data_exploded_2.csv (59, 16) | contributing rows: 59 | signed contributing rows: 59



                                              diseases  up  down  majority
cardiovascular disease                              11   9     2  positive
immune system disease                                1   0     1  negative
nervous system disease                              13  11     2  positive
sign or symptom                                      1   0     1  negative
infectious disease                                   5   1     4  negative
pancreas disease                                     2   0     2  negative
urinary system disease                               1   0     1  negative
gastrointestinal disease                             4   0     4  negative
psychiatric disorder                                 1   1     0  positive
disorder of visual system                            7   0     7  negative
musculoskeletal or connective tissue disease         3   0     3  negative
injury, poisoning or other complication              1   1     0  positive
respiratory or thoracic 

## Numbers

In [13]:
print(paper.save_results("variant_pleiotropy", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/variant_pleiotropy.json


,computed
R4.01,20041.000000
R4.02,5595.000000
R4.03,6617.000000
R4.04,120.000000
R4.05,2.140000
R4.06,4766.000000
R4.07,19.000000
R4.08,1.450000
R4.09,0.840000
R4.10,15.073414
